# The counter, on the library

The same round trip as `reference.ipynb`, rebuilt on `wedgie.kernel.Widget`.
Compare the two: the JSON construction, the protocol constants, the comm id
bookkeeping and the display payload have all gone.

Build and publish first:

```
./mill core.jvm.publishLocal
./mill kernel.publishLocal
```

In [ ]:
import $ivy.`io.github.quafadas::wedgie-kernel:0.1.0-SNAPSHOT`

import upickle.default.ReadWriter
import wedgie.EsmSource
import wedgie.kernel.Widget

## The state

An ordinary case class. Its field names become traitlet keys, which is why they
may not start with `_` — the library rejects that at construction rather than
letting it fail silently in the browser.

This type is the thing that is meant to be cross-compiled: the same definition
compiled for the JVM and for Scala.js is what makes the two ends agree by
construction. For now only the kernel side uses it.

In [ ]:
case class Counter(count: Int, label: String) derives ReadWriter

## The widget

`_esm` is still handwritten JavaScript at this stage — replacing it with a
Laminar app is the next phase. `EsmSource.Inline` is the right strategy for 40
lines of JS; see `probes.ipynb` before using it for a real bundle.

In [ ]:
val esm = EsmSource.Inline("""
function render({ model, el }) {
  const label = () => model.get("label");
  const count = () => model.get("count");

  const btn = document.createElement("button");
  const paint = () => { btn.innerHTML = `${label()}: ${count()}`; };
  paint();

  btn.addEventListener("click", () => {
    model.set("count", count() + 1);
    model.save_changes();
  });

  model.on("change:count", paint);
  model.on("change:label", paint);
  el.appendChild(btn);
}
export default { render };
""")

// `commHandler` and `publish` are auto-imported by Almond and resolve as the
// two `using` parameters. If resolution fails, bind them as givens first
// (the ReadWriter context bound shares that clause, so positional args will not work):
//   given almond.interpreter.api.CommHandler  = commHandler
//   given almond.interpreter.api.OutputHandler = publish
val counter = Widget(Counter(0, "clicks"), esm)

## Read the state from Scala

Click the button a few times first. Unlike the reference notebook there is no
buffer to inspect — inbound `update` messages have already been merged into the
state, so this *is* the confirmation that clicks reached the kernel.

In [ ]:
counter.state

## Push state to the browser

`set` diffs against the current state and sends only the keys that changed, so
this puts `{"count": 99}` on the wire, not the whole object.

In [ ]:
counter.set(Counter(99, "clicks"))

In [ ]:
// `modify` is the read-modify-write form, and is atomic with respect to
// inbound messages arriving on the kernel thread.
counter.modify(c => c.copy(count = c.count + 1, label = "bumped"))

## React to the frontend from Scala

This is the point of the whole exercise: a control driving a calculation.

Observers run on a kernel thread, outside any cell execution, so where their
output lands is probe 1 in `probes.ipynb`. Until that is answered, a mutable
buffer read from a later cell is the reliable way to see what happened.

`fromFrontend` distinguishes a human clicking from the `set` calls above —
without it, an observer that calls `set` would loop.

In [ ]:
val log = scala.collection.mutable.ArrayBuffer.empty[String]

counter.onChange { change =>
  if change.fromFrontend then
    log.synchronized { log += s"user set count to ${change.state.count}" }
}

In [ ]:
// Click the button a few more times, then run this.
log.synchronized(log.toList).foreach(println)

## Failures

Handlers are wrapped, so an exception in an observer is recorded rather than
escaping onto a kernel thread where it might take the comm down silently.

In [ ]:
counter.errors.foreach(e => println(e.toString))

## Things the library will not let you do

Both of these fail at construction, before any comm is opened — the failure mode
they replace is a widget that silently never renders.

In [ ]:
// Field names share a flat namespace with the widget's own traitlets.
case class Clashing(_esm: String) derives ReadWriter

scala.util.Try(Widget(Clashing("x"), esm)).failed.foreach(e => println(e.getMessage))

In [ ]:
// State has to serialise to a JSON object, because its fields become the keys.
scala.util.Try(Widget(42, esm)).failed.foreach(e => println(e.getMessage))